# Part 5: Deployment Practice - Build Your First ML API

## 🎯 Learning Objectives
- Understand deployment strategies (Batch vs Real-time)
- **Build a working REST API** with FastAPI
- **Test the API** with real requests

## 🚀 This is HANDS-ON - You'll Build a Real API!

By the end of this notebook, you'll have a working API that serves sentiment predictions!

## Part A: Deployment Strategies (Quick Theory)

### 🎯 Two Main Approaches

#### 1. Batch Processing (Our Use Case)
**What**: Process all products once per night

```
Every night at 2 AM:
1. Load new reviews from last 24 hours
2. Run sentiment model
3. Aggregate by product
4. Update discount recommendations
```

**Pros**: ✅ Simple, ✅ Cost-effective, ✅ Good for daily pricing
**Cons**: ❌ Not real-time

#### 2. Real-Time API (What We'll Build!)
**What**: Predict sentiment instantly when requested

```
User sends review → API → Model predicts → Response
```

**Pros**: ✅ Instant response, ✅ Interactive
**Cons**: ❌ More complex, ❌ Higher cost

**When to use**: Fraud detection, dynamic pricing, chatbots

---

## Part B: Build Your ML API with FastAPI

### What is FastAPI?
- Modern Python web framework
- Built for APIs (like Flask, but faster)
- Automatic documentation

### Our API Will:
1. Load our trained model
2. Accept review text
3. Return sentiment + discount recommendation

Let's build it!

### Step 1: Install Dependencies

In [ ]:
# Check if FastAPI is installed
try:
    import fastapi
    import uvicorn
    print("✅ FastAPI already installed")
except ImportError:
    print("Installing FastAPI...")
    !pip install fastapi uvicorn pydantic --quiet
    print("✅ FastAPI installed successfully")

### Step 2: Create the API

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict
import pickle
import uvicorn
from threading import Thread
import time

# Load trained model and vectorizer
print("Loading model..")

with open('../models/sentiment_model_v1.0.0.pkl', 'rb') as f:
    model = pickle.load(f)

with open('../models/tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

print("✅ Model loaded successfully")

# Create FastAPI app
app = FastAPI(
    title="AH Sentiment Analysis API",
    description="Predict sentiment and recommend discounts for product reviews",
    version="1.0.0"
)

# Define request/response models
class ReviewRequest(BaseModel):
    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "review_text": "worst spinach ever, completely rotten"
            }
        }
    )

    review_text: str

class PredictionResponse(BaseModel):
    sentiment: str
    confidence: float
    recommended_discount: float

# API endpoints
@app.get("/")
def root():
    """Welcome message"""
    return {
        "message": "Welcome to AH Sentiment Analysis API",
        "version": "1.0.0",
        "endpoints": {
            "/predict": "POST - Predict sentiment for a review",
            "/health": "GET - Check API health",
            "/docs": "GET - Interactive API documentation"
        }
    }

@app.post("/predict", response_model=PredictionResponse)
def predict_sentiment(request: ReviewRequest):
    """
    Predict sentiment for a product review.
    
    Returns:
    - sentiment: positive, neutral, or negative
    - confidence: probability of the prediction (0-1)
    - recommended_discount: discount percentage (0-1)
    """
    try:
        # Vectorize the input text
        X = vectorizer.transform([request.review_text])
        
        # Predict sentiment
        sentiment = model.predict(X)[0]
        probabilities = model.predict_proba(X)[0]
        confidence = float(max(probabilities))
        
        # Calculate smart discount (using sentiment + confidence)
        # Note: In production, we'd also load stock/sales data from database
        # For demo, we use sentiment-based calculation with confidence adjustment
        if sentiment == 'negative':
            discount = 0.50 + (confidence * 0.30)  # 50-80% for negative
        elif sentiment == 'neutral':
            discount = 0.20 + (confidence * 0.10)  # 20-30% for neutral
        else:  # positive
            discount = 0.00 + (confidence * 0.10)  # 0-10% for positive
        
        return PredictionResponse(
            sentiment=sentiment,
            confidence=confidence,
            recommended_discount=discount
        )
        
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction error: {str(e)}")

@app.get("/health")
def health_check():
    """Health check endpoint for monitoring"""
    return {
        "status": "healthy",
        "model_version": "1.0.0",
        "model_loaded": model is not None,
        "vectorizer_loaded": vectorizer is not None
    }

print("✅ API created successfully")
print("\nℹ️  API Endpoints:")
print("  - GET  /          : Welcome message")
print("  - POST /predict   : Predict sentiment")
print("  - GET  /health    : Health check")
print("  - GET  /docs      : Interactive documentation")

### Step 3: Start the API Server

In [ ]:
# Run server in background thread (for notebook demo)
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

# Start server
server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(1)

print("🚀 API Server Started!")
print("="*60)
print("📍 API URL: http://127.0.0.1:8000")
print("📖 Interactive Docs: http://127.0.0.1:8000/docs")
print("="*60)
print("\n✅ Server is running. You can now test it below!")

---

## Part C: Test the API

Now let's send requests to our API and see it work!

### Test 1: Negative Review

In [ ]:
import requests

# Test negative review
response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={"review_text": "worst spinach ever, completely rotten and disgusting"}
)

print("❌ Test: Negative Review")
print("="*60)
print(f"Review: 'worst spinach ever, completely rotten and disgusting'")
print(f"\nAPI Response:")
result = response.json()
print(f"  Sentiment: {result['sentiment']}")
print(f"  Confidence: {result['confidence']:.1%}")
print(f"  Recommended Discount: {result['recommended_discount']:.0%}")
print("="*60)

### Test 2: Positive Review

In [ ]:
# Test positive review
response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={"review_text": "amazing potatoes! Best quality I've ever had"}
)

print("✅ Test: Positive Review")
print("="*60)
print(f"Review: 'amazing potatoes! Best quality I've ever had'")
print(f"\nAPI Response:")
result = response.json()
print(f"  Sentiment: {result['sentiment']}")
print(f"  Confidence: {result['confidence']:.1%}")
print(f"  Recommended Discount: {result['recommended_discount']:.0%}")
print("="*60)

### Test 3: Neutral Review

In [ ]:
# Test neutral review
response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={"review_text": "carrot was okay, nothing special"}
)

print("⚪ Test: Neutral Review")
print("="*60)
print(f"Review: 'carrot was okay, nothing special'")
print(f"\nAPI Response:")
result = response.json()
print(f"  Sentiment: {result['sentiment']}")
print(f"  Confidence: {result['confidence']:.1%}")
print(f"  Recommended Discount: {result['recommended_discount']:.0%}")
print("="*60)

### Test 4: Health Check

In [ ]:
# Test health endpoint
response = requests.get("http://127.0.0.1:8000/health")

print("🏥 Health Check")
print("="*60)
health = response.json()
for key, value in health.items():
    print(f"  {key}: {value}")
print("="*60)

### 🎯 Try it yourself!

**Challenge**: Test the API with your own review text!

In [ ]:
# YOUR TURN: Test with your own review
your_review = "fresh tomatoes, very tasty"  # Change this!

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={"review_text": your_review}
)

print(f"Your Review: '{your_review}'")
print(f"\nPrediction:")
result = response.json()
print(f"  Sentiment: {result['sentiment']}")
print(f"  Confidence: {result['confidence']:.1%}")
print(f"  Recommended Discount: {result['recommended_discount']:.0%}")

---

## Part D: Production Deployment (Theory)

### What We Built vs Production at Albert Heijn

| Component | Our Workshop | AH Production |
|-----------|--------------|---------------|
| **API Framework** | FastAPI (local) | FastAPI on **Azure App Service** |
| **Model Storage** | Local .pkl files | **Azure Blob Storage** |
| **Model Registry** | Python dict | **Azure ML Model Registry** |
| **Hosting** | Localhost | **Azure Kubernetes Service (AKS)** |
| **Monitoring** | Print statements | **Prometheus + Grafana** |
| **Scaling** | Single instance | **Auto-scaling** (handles 1000s requests/sec) |
| **Security** | None | **OAuth2, API keys, rate limiting** |

### Production Architecture

```
Customer/Store System
       ↓
Azure API Management (authentication, rate limiting)
       ↓
Azure Kubernetes Service (AKS)
├── FastAPI containers (3+ replicas)
├── Load balancer
└── Auto-scaling (CPU/memory based)
       ↓
Azure Blob Storage (model files)
       ↓
Azure ML (model registry, monitoring)
       ↓
Prometheus + Grafana (real-time dashboards)
```

### Deployment Pipeline (CI/CD)

```yaml
# When code is pushed to GitHub:
1. Run tests (unit, integration)
2. Build Docker image
3. Push to Azure Container Registry
4. Deploy to staging environment
5. Run smoke tests
6. If tests pass → Deploy to production
7. Monitor for 24 hours
8. Rollback if issues detected
```

### Cost & Scale

**Our workshop**: Free (runs locally)

**AH Production**:
- **Monthly cost**: ~€500-1000 (API hosting + monitoring)
- **Handles**: 10,000+ requests/hour
- **Uptime**: 99.9% (8 hours downtime/year max)
- **ROI**: €50M savings/year → Worth every cent! 🚀

---

## 💡 Summary: What You Learned

### 🚀 Complete Workshop Journey

**Notebook 1**: Business Analysis → Validated project feasibility

**Notebook 2**: ETL & Data Engineering → Bronze → Silver → Gold pipeline

**Notebook 3**: ML Model Training → Trained model + versioning + validation ✅

**Notebook 4** (this one): Deployment → Built working API! 🎉

### 💼 What This Means for You

You now understand the **complete ML lifecycle**:
- ✅ Data Analysis & Business Validation
- ✅ Data Engineering (ETL pipelines)
- ✅ ML Model Engineering (train, evaluate)
- ✅ MLOps (versioning, validation, registry)
- ✅ Deployment (APIs, monitoring)

**This is what companies hire for!**

### 🌟 **Questions? Discussion? Let's talk!** 💬






---

## 🛑 Stop the API Server

**Important**: When you're done, stop the API server to free up resources.

The server is running in the background. We need to stop it properly.

In [ ]:
import psutil

def stop_api_server():
    """Stop the FastAPI server running on port 8000"""
    print("🛑 Stopping API server...")
    print("=" * 70)
    
    stopped = False
    
    # Find process using port 8000
    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        try:
            # Check if process is using port 8000
            cmdline = proc.info['cmdline']
            if cmdline and ('uvicorn' in ' '.join(cmdline) or '8000' in ' '.join(cmdline)):
                print(f"Found API process: PID {proc.info['pid']}")
                proc.terminate()  # Graceful shutdown
                proc.wait(timeout=5)  # Wait for it to stop
                print(f"✅ Stopped API server (PID {proc.info['pid']})")
                stopped = True
                break
        except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.TimeoutExpired):
            continue
    
    if not stopped:
        print("ℹ️  No API server found running on port 8000")
        print("   (It may have already been stopped)")
    
    print("=" * 70)
    print("✅ Done!")

# Run the stop function
stop_api_server()

### Alternative: Stop Server Manually

If the above doesn't work, you can stop it manually:

**Option 1**: Restart Jupyter kernel (Kernel → Restart)

**Option 2**: Find and kill the process:
```bash
# Find the process
lsof -i :8000

# Kill it (replace PID with the actual process ID)
kill -9 PID
```
